# NB01 - Baselines Fortes: B1, B2, B3 (BTCS)
**Andre da Costa Silva | ITA | 2026**

- **B1 Temporal-WCC**: quebra componentes com gap temporal > delta
- **B2 WCC+Ctx**: adiciona arestas do periodo completo dentro do intervalo do caso
- **B3 Hub-Pruned**: remove arestas com nos de grau > percentil H antes do WCC

Ordem de execucao: celulas 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> 9

In [ ]:
# CELULA 1 - Setup: imports, paths, verificacao
import os, sys, subprocess, time, math, contextlib
from pathlib import Path

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'psutil', 'tqdm')
try:
    import torch
except ImportError:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
    import torch

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | device: {DEVICE}')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f'[WARN] Drive: {e}')

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()

# Caminhos confirmados marco/2026
AML100K_BASE  = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k'
AML100K_ARTIF = AML100K_BASE / 'artifacts'
AML100K_PROBS = AML100K_BASE / 'results/probs_v4'
AML100K_MODEL = 'SAGE'
AML100K_SEED  = 42

AML1M_BASE    = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M'
AML1M_ARTIF   = AML1M_BASE / 'artifacts'
AML1M_PROBS   = AML1M_BASE / 'results_aml1m_graphsage_only/probs_v4'
AML1M_MODEL   = 'GraphSAGE'
AML1M_SEED    = 44

BTCS_OUT = BASE / 'GrafosGNN/results/nb01_baselines'
BTCS_OUT.mkdir(parents=True, exist_ok=True)

print('\n=== Verificacao de arquivos ===')
for label, path in [
    ('AML100k cache', AML100K_ARTIF / 'edge_data_v4_clean.pt'),
    ('AML100k probs', AML100K_PROBS / f'{AML100K_MODEL}_seed{AML100K_SEED}_test.npz'),
    ('AML1M   cache', AML1M_ARTIF   / 'edge_data_v4_clean.pt'),
    ('AML1M   probs', AML1M_PROBS   / f'{AML1M_MODEL}_seed{AML1M_SEED}_test.npz'),
]:
    print(f"  {'ok' if path.exists() else 'MISSING'}  {label}")


In [ ]:
# CELULA 2 - Funcoes base (measure_resources, Union-Find, WCC, evaluate_cases, load_artifacts)

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    m0   = proc.memory_info().rss / 1024**2
    t0   = time.perf_counter()
    r    = {}
    yield r
    r['time_s'] = time.perf_counter() - t0
    r['ram_mb']  = proc.memory_info().rss / 1024**2 - m0

def _uf_init(n):
    return np.arange(n, dtype=np.int64), np.zeros(n, dtype=np.int64)

def _find(p, a):
    while p[a] != a:
        p[a] = p[p[a]]; a = p[a]
    return a

def _union(p, r, a, b):
    ra, rb = _find(p,a), _find(p,b)
    if ra==rb: return
    if r[ra]<r[rb]: p[ra]=rb
    elif r[ra]>r[rb]: p[rb]=ra
    else: p[rb]=ra; r[ra]+=1

def _build_wcc(src_sel, dst_sel, sel_idx, full_edges, min_nodes=3):
    nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap  = {int(n):i for i,n in enumerate(nodes)}
    N     = len(nodes)
    us = np.fromiter((nmap[int(x)] for x in src_sel), dtype=np.int64)
    vs = np.fromiter((nmap[int(x)] for x in dst_sel), dtype=np.int64)
    p, r = _uf_init(N)
    for a,b in zip(us,vs): _union(p,r,int(a),int(b))
    comp = np.array([_find(p,i) for i in range(N)], dtype=np.int64)
    _, cids = np.unique(comp, return_inverse=True)
    sizes = np.bincount(cids)
    valid = np.where(sizes >= min_nodes)[0]
    gremap = {int(c):i for i,c in enumerate(valid)}
    maxn   = int(nodes.max())+1
    gof    = -np.ones(maxn, dtype=np.int64)
    for nid, li in nmap.items():
        c = int(cids[li])
        if c in gremap: gof[nid] = gremap[c]
    nc = len(valid)
    if nc == 0: return []
    cases = [{'nodes':set(),'seed_edges':[],'induced_edges':[]} for _ in range(nc)]
    for i,(u,v) in enumerate(zip(src_sel,dst_sel)):
        g = gof[int(u)] if int(u)<maxn else -1
        if g>=0:
            cases[g]['seed_edges'].append(int(sel_idx[i]))
            cases[g]['nodes'].update([int(u),int(v)])
    sf = np.asarray(full_edges['src'], dtype=np.int64)
    df = np.asarray(full_edges['dst'], dtype=np.int64)
    for i,(u,v) in enumerate(zip(sf,df)):
        gu = gof[int(u)] if int(u)<maxn else -1
        gv = gof[int(v)] if int(v)<maxn else -1
        if gu>=0 and gu==gv: cases[gu]['induced_edges'].append(i)
    return cases

def evaluate_cases(cases, gt, ranked, k):
    y_full = np.asarray(gt['y_full'], dtype=int)
    y_test = np.asarray(ranked['y'],  dtype=int)
    pos_te = int(y_test.sum())
    K      = max(1, int(math.ceil(k*len(y_test))))
    if not cases:
        return {m:np.nan for m in ['n_cases','coverage','purity_induced','cr_at_k',
                'edges_per_case_median','edges_per_case_p95','edges_per_case_max',
                'e_ind_over_ek','ocr_b100','ocr_b500','ocr_b1000']}
    ind  = np.array([len(c['induced_edges']) for c in cases])
    ppos = sum(int(y_full[c['induced_edges']].sum()) for c in cases)
    cov  = float(ppos/max(pos_te,1))
    pur  = float(ppos/max(ind.sum(),1))
    psel = sum(int(y_test[c['seed_edges']].sum()) for c in cases)
    rec  = float(psel/max(pos_te,1))
    crk  = float(rec/max(cov,1e-12)) if cov>0 else np.nan
    return {'n_cases':len(cases),'coverage':cov,'purity_induced':pur,'cr_at_k':crk,
            'recall_in_cases':rec,
            'edges_per_case_median':float(np.median(ind)),
            'edges_per_case_p95':float(np.percentile(ind,95)),
            'edges_per_case_max':float(ind.max()),
            'e_ind_total':int(ind.sum()),
            'e_ind_over_ek':float(ind.sum()/max(K,1)),
            'ocr_b100':float((ind>100).mean()),
            'ocr_b500':float((ind>500).mean()),
            'ocr_b1000':float((ind>1000).mean())}

def load_dataset_artifacts(artif_dir, probs_dir, model, seed, dataset_name=''):
    artif_dir = Path(artif_dir); probs_dir = Path(probs_dir)
    npz = np.load(probs_dir / f'{model}_seed{seed}_test.npz')
    p_te = np.asarray(npz['p'], dtype=float)
    y_te = np.asarray(npz['y'], dtype=int)
    print(f'[{dataset_name}] {len(p_te):,} arestas de teste, {y_te.sum():,} positivas ({100*y_te.mean():.2f}%)')
    cache  = torch.load(artif_dir/'edge_data_v4_clean.pt', map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    src_te = ei_all[0, te_idx]; dst_te = ei_all[1, te_idx]
    assert len(p_te)==len(src_te), f'Mismatch scores/arestas: {len(p_te)} vs {len(src_te)}'
    print(f'[{dataset_name}] Cache: {ei_all.shape[1]:,} arestas totais, {len(te_idx):,} no teste')
    ranked = {'scores':p_te,'y':y_te,'src':src_te,'dst':dst_te}
    full   = {'src':src_te,'dst':dst_te}
    gt     = {'y_full':y_te}
    return ranked, full, gt, te_idx, ei_all   # 5 valores sempre

def load_timestamps_from_csv(data_dir, te_idx, ei_all):
    data_dir = Path(data_dir)
    candidates = ['transactions.csv','transaction.csv',
                  'hi-large_trans.csv','hi-medium_trans.csv','hi-small_trans.csv',
                  'li-large_trans.csv','li-medium_trans.csv','li-small_trans.csv']
    csv_path = next((list(data_dir.rglob(c))[0] for c in candidates
                     if list(data_dir.rglob(c))), None)
    if csv_path is None:
        raise FileNotFoundError(f'CSV nao encontrado em {data_dir}')
    print(f'  CSV: {csv_path.name}')
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    time_col = next((c for c in ['timestamp','time','date','datetime','step'] if c in df.columns), None)
    src_col  = next((c for c in ['sender_account_id','src','source','from','sender','src_id'] if c in df.columns), None)
    dst_col  = next((c for c in ['receiver_account_id','dst','dest','to','receiver','dst_id'] if c in df.columns), None)
    if time_col is None:
        raise KeyError(f'Coluna de tempo nao encontrada. Colunas disponiveis: {list(df.columns)}')
    print(f'  Colunas: time={time_col!r}  src={src_col!r}  dst={dst_col!r}')
    # Converter timestamps (igual Stage 1)
    if pd.api.types.is_numeric_dtype(df[time_col]):
        ts_raw = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(np.int64).values
    else:
        ts_raw = (pd.to_datetime(df[time_col], errors='coerce')
                    .fillna(pd.Timestamp('1970-01-01')).astype(np.int64)).values
    # Sort por timestamp (igual Stage 1: sort_values('_ts'))
    order   = np.argsort(ts_raw, kind='stable')
    ts_sort = ts_raw[order]
    # Remover self-loops (igual Stage 1: mask_valid = src_idx != dst_idx)
    if src_col and dst_col:
        src_arr = df[src_col].astype(str).values[order]
        dst_arr = df[dst_col].astype(str).values[order]
        mask    = src_arr != dst_arr
        ts_clean = ts_sort[mask]
        n_loops  = int((~mask).sum())
        if n_loops > 0:
            print(f'  Self-loops removidos: {n_loops}')
    else:
        ts_clean = ts_sort
    n_csv, n_cache = len(ts_clean), ei_all.shape[1]
    if n_csv != n_cache:
        raise AssertionError(f'Mismatch: {n_csv} timestamps vs {n_cache} arestas. Diferenca: {n_csv-n_cache}')
    ts_test = ts_clean[te_idx]
    print(f'  Timestamps: range [{ts_test.min()}, {ts_test.max()}]')
    return ts_test

print('Funcoes base definidas.')


In [ ]:
# CELULA 3 - B1 (Temporal-WCC), B2 (WCC+Ctx), B3 (Hub-Pruned)

# ---- B1: Temporal-WCC ----
# WCC padrao + quebra componentes onde gap temporal entre arestas > delta_steps
def build_b1_temporal_wcc(ranked, full, k=0.01, delta_steps=7, min_nodes=3):
    scores = np.asarray(ranked['scores'], dtype=float)
    src    = np.asarray(ranked['src'],    dtype=np.int64)
    dst    = np.asarray(ranked['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked['timestamps'], dtype=np.int64)
    K      = max(1, int(math.ceil(k*len(scores))))
    sel    = np.argsort(-scores)[:K]
    wcc    = _build_wcc(src[sel], dst[sel], sel, full, min_nodes=1)
    sf     = np.asarray(full['src'], dtype=np.int64)
    df_    = np.asarray(full['dst'], dtype=np.int64)
    ts_f   = np.asarray(full['timestamps'], dtype=np.int64)
    out    = []
    for case in wcc:
        idx  = np.array(case['seed_edges'], dtype=np.int64)
        if len(idx)==0: continue
        order    = np.argsort(ts[idx])
        ts_s     = ts[idx][order]
        idx_s    = idx[order]
        gaps     = np.diff(ts_s)
        breaks   = np.where(gaps > delta_steps)[0]+1
        for chunk in np.split(idx_s, breaks):
            if len(chunk)==0: continue
            ns = set(src[chunk].tolist())|set(dst[chunk].tolist())
            if len(ns)<min_nodes: continue
            t0_,t1_ = int(ts[chunk].min()), int(ts[chunk].max())
            ind = [i for i,(u,v,t) in enumerate(zip(sf,df_,ts_f))
                   if int(u) in ns and int(v) in ns and t0_<=t<=t1_]
            out.append({'nodes':ns,'seed_edges':list(chunk),'induced_edges':ind})
    return out

# ---- B2: WCC + Contexto Temporal ----
# WCC padrao + expande janela de tempo de cada caso por delta_ctx
def build_b2_wcc_ctx(ranked, full, k=0.01, delta_ctx=3, min_nodes=3):
    scores = np.asarray(ranked['scores'], dtype=float)
    src    = np.asarray(ranked['src'],    dtype=np.int64)
    dst    = np.asarray(ranked['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked['timestamps'], dtype=np.int64)
    K      = max(1, int(math.ceil(k*len(scores))))
    sel    = np.argsort(-scores)[:K]
    wcc    = _build_wcc(src[sel], dst[sel], sel, full, min_nodes=min_nodes)
    sf     = np.asarray(full['src'], dtype=np.int64)
    df_    = np.asarray(full['dst'], dtype=np.int64)
    ts_f   = np.asarray(full['timestamps'], dtype=np.int64)
    out    = []
    for case in wcc:
        idx  = np.array(case['seed_edges'], dtype=np.int64)
        if len(idx)==0: continue
        t0_  = int(ts[idx].min())-delta_ctx
        t1_  = int(ts[idx].max())+delta_ctx
        ns   = case['nodes']
        ctx  = set()
        for u,v,t in zip(sf,df_,ts_f):
            if t0_<=t<=t1_ and (int(u) in ns or int(v) in ns):
                ctx.add(int(u)); ctx.add(int(v))
        exp  = ns|ctx
        ind  = [i for i,(u,v,t) in enumerate(zip(sf,df_,ts_f))
                if int(u) in exp and int(v) in exp and t0_<=t<=t1_]
        out.append({'nodes':exp,'seed_edges':list(idx),'induced_edges':ind})
    return out

# ---- B3: Hub-Pruned WCC ----
# Remove arestas com nos de grau > percentil hub_pct antes do WCC
def build_b3_hub_pruned(ranked, full, k=0.01, hub_pct=95, min_nodes=3):
    scores = np.asarray(ranked['scores'], dtype=float)
    src    = np.asarray(ranked['src'],    dtype=np.int64)
    dst    = np.asarray(ranked['dst'],    dtype=np.int64)
    K      = max(1, int(math.ceil(k*len(scores))))
    sel    = np.argsort(-scores)[:K]
    sf     = np.asarray(full['src'], dtype=np.int64)
    df_    = np.asarray(full['dst'], dtype=np.int64)
    maxn   = int(max(sf.max(), df_.max()))+1
    deg    = np.zeros(maxn, dtype=np.int64)
    for u,v in zip(sf,df_): deg[u]+=1; deg[v]+=1
    H      = np.percentile(deg[deg>0], hub_pct)
    ss,ds  = src[sel], dst[sel]
    mask   = (deg[ss]<=H)&(deg[ds]<=H)
    nrem   = int((~mask).sum())
    print(f'  B3: H={H:.0f} | {nrem}/{K} arestas hub removidas ({100*nrem/K:.1f}%)')
    if mask.sum()==0: return []
    return _build_wcc(ss[mask], ds[mask], sel[mask], full, min_nodes=min_nodes)

print('B1, B2, B3 definidos.')


In [ ]:
# CELULA 4 - Carregar artefatos + timestamps (executar uma vez)

print('Carregando AML100k...')
ranked_100k, full_100k, gt_100k, te_idx_100k, ei_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS, AML100K_MODEL, AML100K_SEED, 'AML100k')
ts_100k = load_timestamps_from_csv(AML100K_BASE, te_idx_100k, ei_100k)
ranked_100k['timestamps'] = ts_100k
full_100k['timestamps']   = ts_100k

print('\nCarregando AML1M (pode demorar ~1-2 min)...')
ranked_1m, full_1m, gt_1m, te_idx_1m, ei_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS, AML1M_MODEL, AML1M_SEED, 'AML1M')
ts_1m = load_timestamps_from_csv(AML1M_BASE, te_idx_1m, ei_1m)
ranked_1m['timestamps'] = ts_1m
full_1m['timestamps']   = ts_1m

print('\nDados + timestamps prontos!')


In [ ]:
# CELULA 5 - Hiperparametros (tunar em val, NAO no teste)
B1_DELTA     = 7     # gap max em steps. Sweep: {1,3,7,14,30}
B2_DELTA_CTX = 3     # contexto em steps. Sweep: {1,3,7,14}
B3_HUB_PCT   = 95    # percentil de grau. Sweep: {90,95,97,99}
KS           = [0.01, 0.02, 0.05, 0.10]
MIN_NODES    = 3

print(f'B1: delta={B1_DELTA}  B2: ctx={B2_DELTA_CTX}  B3: hub_pct={B3_HUB_PCT}')
print(f'k values: {[str(int(k*100))+"%" for k in KS]}')


In [ ]:
# CELULA 6 - Rodar B0 + B1 + B2 + B3
DATASETS = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]
rows = []

for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*50}  {ds}  {"="*50}')
    for k in KS:
        kp = f'{int(k*100)}%'
        K_ = max(1, int(math.ceil(k*len(ranked['scores']))))
        sc = np.asarray(ranked['scores']); sr = np.asarray(ranked['src']); dr = np.asarray(ranked['dst'])
        sel_ = np.argsort(-sc)[:K_]
        print(f'\n  k={kp}:')

        with measure_resources() as res:
            cases = _build_wcc(sr[sel_],dr[sel_],sel_,full,min_nodes=MIN_NODES)
            m = evaluate_cases(cases,gt,ranked,k)
        rows.append({'dataset':ds,'method':'B0_WCC','k%':kp,'k_frac':k,**m,**res})
        print(f"    B0 WCC:   #casos={m['n_cases']:,} cov={m['coverage']:.3f} purity={m['purity_induced']:.4f} OCR100={m['ocr_b100']:.3f} E/Ek={m['e_ind_over_ek']:.2f}x {res['time_s']:.1f}s")

        with measure_resources() as res:
            cases = build_b1_temporal_wcc(ranked,full,k=k,delta_steps=B1_DELTA,min_nodes=MIN_NODES)
            m = evaluate_cases(cases,gt,ranked,k)
        rows.append({'dataset':ds,'method':'B1_TemporalWCC','k%':kp,'k_frac':k,**m,**res,'delta':B1_DELTA})
        print(f"    B1 Temp:  #casos={m['n_cases']:,} cov={m['coverage']:.3f} purity={m['purity_induced']:.4f} OCR100={m['ocr_b100']:.3f} E/Ek={m['e_ind_over_ek']:.2f}x {res['time_s']:.1f}s")

        with measure_resources() as res:
            cases = build_b2_wcc_ctx(ranked,full,k=k,delta_ctx=B2_DELTA_CTX,min_nodes=MIN_NODES)
            m = evaluate_cases(cases,gt,ranked,k)
        rows.append({'dataset':ds,'method':'B2_WCC_Ctx','k%':kp,'k_frac':k,**m,**res,'delta':B2_DELTA_CTX})
        print(f"    B2 Ctx:   #casos={m['n_cases']:,} cov={m['coverage']:.3f} purity={m['purity_induced']:.4f} OCR100={m['ocr_b100']:.3f} E/Ek={m['e_ind_over_ek']:.2f}x {res['time_s']:.1f}s")

        with measure_resources() as res:
            cases = build_b3_hub_pruned(ranked,full,k=k,hub_pct=B3_HUB_PCT,min_nodes=MIN_NODES)
            m = evaluate_cases(cases,gt,ranked,k)
        rows.append({'dataset':ds,'method':'B3_HubPruned','k%':kp,'k_frac':k,**m,**res,'hub_pct':B3_HUB_PCT})
        print(f"    B3 Hub:   #casos={m['n_cases']:,} cov={m['coverage']:.3f} purity={m['purity_induced']:.4f} OCR100={m['ocr_b100']:.3f} E/Ek={m['e_ind_over_ek']:.2f}x {res['time_s']:.1f}s")

df_results = pd.DataFrame(rows)
print('\nConcluido!')


In [ ]:
# CELULA 7 - Tabela comparativa + analise go/no-go

for ds in ['AML100k','AML1M']:
    pivot = df_results[df_results['dataset']==ds].pivot_table(
        index='k%', columns='method',
        values=['n_cases','coverage','purity_induced','ocr_b100','e_ind_over_ek'],
        aggfunc='first').round(4)
    print(f'\n=== {ds} ===')
    display(pivot)

print('\n' + '='*60)
print('ANALISE GO/NO-GO: B1 vs B0 (AML1M, k=5% e k=10%)')
print('Criterio: reducao de OCR-B(100) >= 70% = GO')
print('='*60)

go = False
for kpct in ['5%','10%']:
    b0 = df_results[(df_results['dataset']=='AML1M')&(df_results['method']=='B0_WCC')&(df_results['k%']==kpct)]
    b1 = df_results[(df_results['dataset']=='AML1M')&(df_results['method']=='B1_TemporalWCC')&(df_results['k%']==kpct)]
    if b0.empty or b1.empty: continue
    ocr0 = float(b0.iloc[0]['ocr_b100'])
    ocr1 = float(b1.iloc[0]['ocr_b100'])
    red  = (ocr0-ocr1)/max(ocr0,1e-9)
    s    = 'GO' if red>=0.70 else ('PARCIAL' if red>=0.40 else 'NO-GO')
    if red>=0.70: go=True
    print(f'  k={kpct}: OCR0={ocr0:.3f} -> OCR1={ocr1:.3f}  reducao={100*red:.1f}%  [{s}]')

print()
if go:
    print('-> B1 resolve parte significativa. Considerar mudar narrativa.')
else:
    print('-> Temporal puro insuficiente. Narrativa: BTCS vs WCC (B0).')
    print('   B1/B2/B3 entram como ablacoes na Tabela 3.')


In [ ]:
# CELULA 8 - Plot OCR-B comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
METHODS = ['B0_WCC','B1_TemporalWCC','B2_WCC_Ctx','B3_HubPruned']
COLORS  = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
LABELS  = ['B0 WCC','B1 Temporal','B2 WCC+Ctx','B3 Hub-Pruned']

for ax, ds in zip(axes, ['AML100k','AML1M']):
    dfd = df_results[df_results['dataset']==ds]
    x   = np.arange(len(KS)); w = 0.18
    for j,(m,col,lab) in enumerate(zip(METHODS,COLORS,LABELS)):
        dfm = dfd[dfd['method']==m]
        vals = [float(dfm[dfm['k%']==f'{int(k*100)}%']['ocr_b100'].values[0])
                if not dfm[dfm['k%']==f'{int(k*100)}%'].empty else 0 for k in KS]
        ax.bar(x+j*w, vals, w, label=lab, color=col, alpha=0.85)
    ax.set_title(f'{ds} - OCR(B=100)')
    ax.set_xticks(x+1.5*w); ax.set_xticklabels([f'{int(k*100)}%' for k in KS])
    ax.set_ylabel('fracao casos > 100 arestas'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(BTCS_OUT/'ocr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot salvo.')


In [ ]:
# CELULA 9 - Export CSV + LaTeX
df_results.to_csv(BTCS_OUT/'b0_b1_b2_b3_results.csv', index=False)
print(f'CSV salvo: {BTCS_OUT}/b0_b1_b2_b3_results.csv')

cols = ['dataset','method','k%','n_cases','coverage','purity_induced','cr_at_k','e_ind_over_ek','ocr_b100','time_s']
df_results[cols].round(4).to_latex(BTCS_OUT/'b0_b1_b2_b3_table.tex', index=False, escape=False)
print('LaTeX salvo.')

print('\nRoadmap:')
print('1. [ok] nb00 - WCC reproduzido')
print('2. [ok] nb01 - B1, B2, B3')
print('3. [ ] nb02 - Grafo Lk + Leiden (BTCS)')
print('4. [ ] nb03 - Ablacoes')
print('5. [ ] nb04 - AMLworld + Libra Bank')
